## 0. Setup (run first in every notebook)
Attach these inputs via **Add Input**: your uploaded `causal-rag-kaggle` dataset (this project) and, depending on the stage,
the Lending Club dataset (`wordsforthewise/lending-club`) and/or the artifacts dataset saved from the previous notebook.
Turn **Internet ON** (policy documents, models, RCT data) and set the accelerator to **GPU T4 x2** for stages 04-08.

In [ ]:
!pip -q install causal-learn econml sentence-transformers bitsandbytes accelerate gradio pypdf beautifulsoup4 scikit-uplift pyarrow 2>&1 | tail -1
import sys, os, glob
hits = glob.glob('/kaggle/input/**/causal_rag/__init__.py', recursive=True)
print('causal_rag copies found:'); [print('  ', h) for h in hits]
need = ('data_prep.py', 'causal_graph.py', 'retrieval.py', 'counterfactual.py', 'audit.py', 'compare.py', 'pipeline.py', 'live_demo.py')
complete = [h for h in hits if all(os.path.exists(os.path.join(os.path.dirname(h), n)) for n in need)]
v11 = [h for h in complete if 'prompt_graded' in open(os.path.join(os.path.dirname(h), 'prompts.py')).read()]
if not complete:
    for h in hits: print('FILES in', os.path.dirname(h), '->', sorted(os.listdir(os.path.dirname(h))))
assert complete, "No COMPLETE causal_rag package found in the attached dataset: re-upload causal_rag_kaggle.zip as a new dataset version."
assert v11, "The mounted package is an OLD version (prompts.py has no prompt_graded; need code 1.3.0). Upload the latest causal_rag_kaggle.zip as a NEW VERSION of the dataset, STOP the session (power icon), re-attach the latest version, and start again."
pkg = v11[0]
for m in [m for m in sys.modules if m.startswith('causal_rag')]: del sys.modules[m]   # drop any stale import
sys.path.insert(0, os.path.dirname(os.path.dirname(pkg)))
print('USING:', pkg)
from causal_rag import paths
paths.restore_artifacts()          # pulls artifacts from any attached output-dataset of an earlier stage
SMOKE   = False                    # True = tiny REAL-data dry run (minutes) to verify the pipeline
N_EVAL  = 12 if SMOKE else 600     # applicants scored by the LLM systems (v1.0/v1.1 used 300; ~5 h at 600, ~6.5 h at 800)
CHUNK   = 6  if SMOKE else 100     # LLM stage saves every CHUNK applicants and resumes after a crash
RUN_NAME = 'v1_2_run'               # name of this run's folder in the exported results
AUDIT_SYSTEM = 'causal_rag_smooth'  # system used by the audit agent: 'causal_rag' (v1.1 prompt) or 'causal_rag_smooth'
EPOCHS  = 3  if SMOKE else 40
N_REPORT= 3  if SMOKE else 10      # applicants that get full DAG + HTML + rationale
WITH_ECONML = False                # EconML causal forest is very slow (hours on CPU); turn on only if you have the time
import torch
HAS_GPU = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if HAS_GPU else 'NONE - CPU ONLY')
REQUIRE_GPU = True                 # stages 03-08 need a GPU: on CPU, embeddings alone took ~6 h and CALF ~2.5 h
assert HAS_GPU or not REQUIRE_GPU or SMOKE, ("No GPU in this session. Settings -> Accelerator -> GPU T4 x2 "
                                             "(or your GPU quota is used up). Refusing to start a multi-hour CPU run.")
print('artifacts dir:', paths.ART, '| SMOKE =', SMOKE)

# FINAL v2.0 - last run: band-table experiment + automatic final report
One notebook, one run (about 1.5-2 h on a T4; resumable). It needs the artifacts of your earlier run attached (`evidence.pkl`, `lc_clean.parquet`; optional: `llm_scores.parquet`, `llm_scores_v13.parquet`, `calf_scores.parquet` for comparisons).
1. **`bands_only`**: applicant + unadjusted default-rate-by-band tables. No cutoffs, flags, regime wording, precedent or CALF numbers.
2. **`causal_graded_bands`**: the graded Causal-RAG prompt plus the same band tables.
Both are scored on the same 600 applicants (AUROC) and put through the local cliff test (150 applicants). Then the notebook writes
`FINAL_REPORT.html / .md` with every table and figure from ALL archived runs plus this one, and zips everything.

In [ ]:
import pickle, json, numpy as np, pandas as pd
need = ['evidence.pkl', 'lc_clean.parquet']
missing = [n for n in need if not paths.have(n)]
assert not missing, f"Missing {missing}: attach the notebook output / dataset holding your run's artifacts/ folder."
from causal_rag import pipeline as pl, eval as ev, compare, report, prompts as pr
from causal_rag.llm import LLM
from causal_rag.config import fit_medians, add_features
bundles = pickle.load(open(paths.art('evidence.pkl'), 'rb'))
N_FINAL = 12 if SMOKE else len(bundles); bundles = bundles[:N_FINAL]
N_CLIFF = 6 if SMOKE else 150
df = pd.read_parquet(paths.art('lc_clean.parquet')); tr = (df.role == 'train').values
d = add_features(df, fit_medians(df, tr)); idx = d[d.role.isin(['train', 'val'])].reset_index(drop=True)
tabs = pr.compute_band_tables(idx); pr.set_band_tables(tabs)
json.dump(tabs, open(paths.art('band_tables.json'), 'w'), indent=1)
for f, rows in tabs.items(): print(f, [(lab, round(100*r, 1)) for lab, r, n in rows])

In [ ]:
llm = LLM()
NEW = ['bands_only', 'causal_graded_bands']
sc20, _ = pl.run_llm_stage(llm, bundles, idx, NEW, [], boundary_systems=(), chunk=CHUNK, tag='llm_parts_v20')
sc20['default'] = df.set_index('case_id').default.loc[sc20.case_id].values
sc20.to_parquet(paths.art('llm_scores_v20.parquet'))
w = pl.run_cliff_stage(llm, bundles[:N_CLIFF], NEW, chunk=25 if not SMOKE else 3, tag='cliff_parts_v20')
w.to_parquet(paths.art('cliff_pairs_v20.parquet'))
print('done:', sc20.shape, w.shape)

In [ ]:
import matplotlib; matplotlib.use('Agg')
from causal_rag.config import Featurizer, TREATMENTS
parts = [sc20]
for f in ('llm_scores.parquet', 'llm_scores_v13.parquet'):
    if paths.have(f): parts.append(pd.read_parquet(paths.art(f)))
sc = parts[0]
for p_ in parts[1:]:
    sc = sc.merge(p_.drop(columns=[c for c in p_.columns if c in sc.columns and c != 'case_id']), on='case_id', how='inner')
y = sc.default.values; print('applicants compared:', len(sc), '| defaults:', int(y.sum()))
LLM_SYS = [c for c in sc.columns if c not in ('case_id', 'default')]
scores = {k: sc[k].values for k in LLM_SYS}
if paths.have('calf_scores.parquet'):
    va = (df.role == 'val').values; yy = d.default.values
    X = np.hstack(Featurizer(None).fit(d, tr).transform(d)); te_idx = d.set_index('case_id').index.get_indexer(sc.case_id.values)
    scores['LightGBM (tabular ceiling)'] = ev.gbm_baseline(X[tr], yy[tr], X[va], yy[va], X[te_idx])
    calf = pd.read_parquet(paths.art('calf_scores.parquet')).set_index('case_id').loc[sc.case_id.values]
    scores['CALF factual risk (tabular)'] = np.mean([calf[f'pf_{tk}'].values for tk in TREATMENTS], axis=0)
REF = 'no_rag' if 'no_rag' in scores else NEW[0]
Bn = 300 if SMOKE else 1000
disc = ev.discrimination_table(y, scores, ref=REF, B=Bn); display(disc)
pairs = [('bands_only', REF), ('causal_graded_bands', REF), ('causal_graded_bands', 'causal_graded'),
         ('causal_graded_bands', 'causal_conditions_only'), ('causal_graded_bands', 'bands_only'),
         ('causal_graded_bands', 'causal_rag'), ('bands_only', 'causal_conditions_only')]
pw = ev.pairwise_table(y, scores, pairs, B=Bn); display(pw)
cl = ev.cliff_table(w, B=300 if SMOKE else 1000); clc = ev.cliff_table(w, by_clause=True, B=200 if SMOKE else 500)
display(cl); display(clc.pivot(index='clause', columns='system', values='cliff_index_abs'))
for r in cl.itertuples():
    print(f"{r.system}: cliff index {r.cliff_index_abs:.2f} [{r.ci_lo:.2f}, {r.ci_hi:.2f}] ->", report.verdict_cliff(r.cliff_index_abs, r.ci_lo, r.ci_hi))
for r in disc[disc.system.isin(NEW)].itertuples():
    print(f"{r.system}: AUROC {r.auroc:.3f}, diff vs {REF} {r.d_vs_ref:+.3f} [{r.d_lo:+.3f}, {r.d_hi:+.3f}] ->", report.verdict_auc(r.d_vs_ref, r.d_lo, r.d_hi))

In [ ]:
import shutil, os
from IPython.display import HTML, display
runs = compare.all_run_dirs(); print('archived runs found:', list(runs))
final_dir = paths.ART / 'results_export' / 'v2_0_final'; final_dir.mkdir(parents=True, exist_ok=True)
disc.to_csv(final_dir / 'stage08_discrimination.csv', index=False); pw.to_csv(final_dir / 'stage08_pairwise.csv', index=False)
cl.to_csv(final_dir / 'cliff_table.csv', index=False); clc.to_csv(final_dir / 'cliff_table_by_clause.csv', index=False)
cfg = dict(code_version='2.0.0', N_EVAL=N_FINAL, N_CLIFF=N_CLIFF, systems_new=NEW, compared_with=REF)
report.build_report(str(final_dir / 'report'), runs, dict(disc=disc[disc.system.isin(NEW)], pairwise=pw, cliff=cl, cliff_clause=clc, bands=tabs), cfg)
(final_dir / 'RUN_INFO.md').write_text('# v2.0 final run' + chr(10) + json.dumps(cfg, indent=1))
shutil.make_archive('/kaggle/working/FINAL_RESULTS_v2_0', 'zip', str(paths.ART / 'results_export'))
print('wrote /kaggle/working/FINAL_RESULTS_v2_0.zip  (report + tables + figures)')
display(HTML(open(final_dir / 'report' / 'FINAL_REPORT.html', encoding='utf-8').read()))

### Optional: package this session's artifacts so they are never lost again
Download `crag_artifacts.zip` from the Output panel and upload it as a Kaggle Dataset (e.g. `crag-artifacts`). Attach that dataset in later notebooks: `restore_artifacts()` and the skip-if-exists guards pick everything up.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/crag_artifacts', 'zip', '/kaggle/working', 'artifacts')
print('wrote /kaggle/working/crag_artifacts.zip')